In [1]:
%cd ..

/home/pratham/Desktop/projects/portfolio/RAG-Evaluator


In [2]:
from src.helper import *

/home/pratham/anaconda3/envs/portfolio/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
df = pd.read_csv('transformer_qa.csv')
df.head()

,questions,ground_truths,model_context
0,What does the Transformer use instead of recur...,The Transformer replaces recurrence and convol...,"“We propose a new simple network architecture,..."


In [4]:
response = ResponseLLM()

In [5]:
question = df['questions']
ground_truths = df['ground_truths']
mode_context = df['model_context']
model_answer = df['model_answer']



In [6]:
ds = Dataset.from_csv('vectra_predict.csv')
ds

Dataset({
    features: ['Unnamed: 0', 'questions', 'model_answer', 'model_context', 'ground_truths', 'vectra', 'score'],
    num_rows: 3
})

In [7]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
critic_model = ChatGroq(temperature=0, model_name='gemma2-9b-it')


In [8]:
ds = ds.rename_column('questions', 'question')
ds = ds.rename_column('model_answer', 'answer')
ds = ds.rename_column('model_context', 'contexts')
ds = ds.rename_column('ground_truths','ground_truth')

In [9]:
# # Ensure the required columns are present in ds
# required_columns = ['answer', 'contexts', 'question']
# missing_columns = [col for col in required_columns if col not in ds.column_names]
# if missing_columns:
#     raise ValueError(f"Missing columns in ds: {missing_columns}")
# else:
#     print("All required columns are present in ds:", required_columns)

In [10]:
# ds.data[3]

In [11]:
import ast

# If contexts are stored as string representations of lists, use ast.literal_eval

def ensure_list(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return [x]
ds = ds.map(lambda row: {"contexts": ensure_list(row["contexts"])})

In [12]:
results = evaluate(ds,
                           metrics=[
                                    # faithfullness,
                                    faithfulness,
                                    answer_relevancy,
                                    context_utilization,
                                    # context_precision,
                                    answer_correctness,
                                    context_entity_recall,
                                    answer_similarity],
                           llm=critic_model, embeddings=embeddings)
        

Evaluating: 100%|██████████| 18/18 [02:56<00:00,  9.83s/it]


In [13]:
results = results.to_pandas()
results.to_csv('ragas_results.csv')

In [14]:
df = pd.read_csv('ragas_results.csv')
df

,Unnamed: 0.1,Unnamed: 0,question,answer,contexts,ground_truth,vectra,score,faithfulness,answer_relevancy,context_utilization,answer_correctness,context_entity_recall,answer_similarity
0,0,0,What does the Transformer use instead of recur...,The Transformer relies entirely on an attentio...,['Figure 1: The Transformer - model architectu...,The Transformer replaces recurrence and convol...,Factual,0.98,1.0,0.778246,1.0,0.837386,0.333333,0.949546
1,1,1,How does the Transformer handle sequence order...,"I am sorry, but the provided documents do not ...",['The Transformer uses multi-head attention in...,It uses sinusoidal positional encodings added ...,Hallucinated,0.17,0.0,0.000000,0.0,0.186604,0.000000,0.746416
2,2,2,Why is self-attention more parallelizable than...,"I'm sorry, but the provided context does not c...",['PEpos.\nWe also experimented with using lear...,Because self-attention does not rely on sequen...,Hallucinated,0.10,0.0,0.000000,0.0,0.224672,0.000000,0.898687
